<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Code_Switching_Sequence_Tagger_Pipeline_(PyTorch).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Example image](https://upload.wikimedia.org/wikipedia/commons/0/02/Northeastern_Wordmark.svg)

# Code-Switching Sequence Tagger Pipeline (PyTorch)

Copyright: Prof. Shanu Sushmita

In [2]:
# ============================================================
# 1️⃣ Setup
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re, warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

# ============================================================
# 2️⃣ Example Data (toy code-switching corpus)
# ============================================================
RAW_DATA = [
    ("Hola I am going to la tienda", ["ES", "EN", "EN", "EN", "EN", "ES"]),
    ("She said vamos a comer", ["EN", "EN", "ES", "ES", "ES"]),
    ("Estoy learning English", ["ES", "EN", "EN"]),
]

# ============================================================
# 3️⃣ Vocabulary & Tag Indexing
# ============================================================
def build_vocab(data):
    word_to_ix = {"<PAD>": 0, "<UNK>": 1}
    tag_to_ix = {"<PAD>": 0}
    for sentence, tags in data:
        tokens = re.findall(r"\w+|[^\s\w]", sentence)
        for token in tokens:
            if token not in word_to_ix:
                word_to_ix[token] = len(word_to_ix)
        for tag in tags:
            if tag not in tag_to_ix:
                tag_to_ix[tag] = len(tag_to_ix)
    return word_to_ix, tag_to_ix

WORD_TO_IX, TAG_TO_IX = build_vocab(RAW_DATA)
IX_TO_TAG = {v: k for k, v in TAG_TO_IX.items()}
PAD_INDEX = WORD_TO_IX["<PAD>"]
UNK_INDEX = WORD_TO_IX["<UNK>"]
MAX_SEQ_LEN = 15

# ============================================================
# 4️⃣ Dataset and Collate Function
# ============================================================
class CodeSwitchDataset(Dataset):
    def __init__(self, data, word_to_ix, tag_to_ix, max_len=MAX_SEQ_LEN):
        self.data = data
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence, tags = self.data[idx]
        tokens = re.findall(r"\w+|[^\s\w]", sentence)
        L = min(len(tokens), len(tags))
        tokens = tokens[:L]
        tags = tags[:L]
        input_ids = torch.tensor([self.word_to_ix.get(w, UNK_INDEX) for w in tokens], dtype=torch.long)
        tag_ids = torch.tensor([self.tag_to_ix[t] for t in tags], dtype=torch.long)
        return input_ids, tag_ids

def collate_fn(batch):
    inputs, targets = zip(*batch)
    padded_inputs = nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=PAD_INDEX)
    padded_targets = nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=PAD_INDEX)
    return padded_inputs, padded_targets

train_dataset = CodeSwitchDataset(RAW_DATA, WORD_TO_IX, TAG_TO_IX)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

# ============================================================
# 5️⃣ Model Definition (BiLSTM)
# ============================================================
class BiLSTM_SequenceTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, tagset_size, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        logits = self.fc(lstm_out)
        return logits

VOCAB_SIZE = len(WORD_TO_IX)
TAG_SIZE = len(TAG_TO_IX)
EMBEDDING_DIM = 64
HIDDEN_DIM = 64

model = BiLSTM_SequenceTagger(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, TAG_SIZE, PAD_INDEX).to(device)

# ============================================================
# 6️⃣ Training Setup
# ============================================================
criterion = nn.CrossEntropyLoss(ignore_index=PAD_INDEX)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
NUM_EPOCHS = 10

# ============================================================
# 7️⃣ Training Loop
# ============================================================
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        logits = logits.reshape(-1, TAG_SIZE)
        targets = targets.reshape(-1)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {total_loss:.4f}")

# ============================================================
# 8️⃣ Evaluation (Simple Inference)
# ============================================================
model.eval()
test_sentence = "Ella is going to school"
tokens = re.findall(r"\w+|[^\s\w]", test_sentence)
input_ids = torch.tensor([[WORD_TO_IX.get(w, UNK_INDEX) for w in tokens]], dtype=torch.long).to(device)

with torch.no_grad():
    logits = model(input_ids)
    preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]
pred_tags = [IX_TO_TAG[i] for i in preds]

print("\nSentence:", test_sentence)
print("Predicted Tags:", pred_tags)


Running on: cuda
Epoch 1/10 - Loss: 2.2530
Epoch 2/10 - Loss: 2.1088
Epoch 3/10 - Loss: 1.9682
Epoch 4/10 - Loss: 1.8539
Epoch 5/10 - Loss: 1.7407
Epoch 6/10 - Loss: 1.6661
Epoch 7/10 - Loss: 1.5707
Epoch 8/10 - Loss: 1.4608
Epoch 9/10 - Loss: 1.3192
Epoch 10/10 - Loss: 1.2148

Sentence: Ella is going to school
Predicted Tags: ['EN', 'EN', 'EN', 'EN', 'EN']
